<a href="https://colab.research.google.com/github/DimosAndronoudis/Tacotron2_vs_VITS_Coqui-TTS/blob/main/Tacotron2_vs_VITS_Coqui_TTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


!apt-get install -y espeak-ng
!pip install py-espeak-ng


!pip install --upgrade pip
!pip install TTS librosa matplotlib --quiet


from TTS.api import TTS
import IPython.display as ipd
from google.colab import files
import torch
import os
import ipywidgets as widgets
from IPython.display import display
import librosa
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
import numpy as np

# Load Tacotron2)
tacotron_model = TTS(model_name="tts_models/en/ljspeech/tacotron2-DDC")
tacotron_model.to("cuda" if torch.cuda.is_available() else "cpu")

# Load VITS
vits_model = TTS(model_name="tts_models/en/ljspeech/vits")
vits_model.to("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
text_input = widgets.Text(
    value='This is a voice synthesis comparison using Tacotron2 and VITS.',
    description='Text:',
    layout=widgets.Layout(width='90%')
)

# Button to generate both
generate_button = widgets.Button(description="Compare Models")
output = widgets.Output()


In [21]:
def plot_waveform_and_spectrogram(audio_path, title):
    y, sr = librosa.load(audio_path, sr=None)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    librosa.display.waveshow(y, sr=sr)
    plt.title(f"Waveform - {title}")
    plt.subplot(1, 2, 2)
    D = librosa.amplitude_to_db(librosa.stft(y), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz')
    plt.title(f"Spectrogram - {title}")
    plt.colorbar(format='%+2.0f dB')
    plt.tight_layout()
    plt.show()

def on_generate_clicked(b):
    output.clear_output()
    with output:
        text = text_input.value

        print("Generating with Tacotron2...")
        tacotron_path = "output_tacotron.wav"
        tacotron_model.tts_to_file(text=text, file_path=tacotron_path)

        print("Generating with VITS...")
        vits_path = "output_vits.wav"
        vits_model.tts_to_file(text=text, file_path=vits_path)

        print("\nPlayback - Tacotron2:")
        display(ipd.Audio(tacotron_path))
        plot_waveform_and_spectrogram(tacotron_path, "Tacotron2")

        print("\nPlayback - VITS:")
        display(ipd.Audio(vits_path))
        plot_waveform_and_spectrogram(vits_path, "VITS")

        print("\nDownload links:")
        files.download(tacotron_path)
        files.download(vits_path)

In [ ]:

generate_button.on_click(on_generate_clicked)

display(text_input, generate_button, output)